# Vision Transformer：从 Patch Embedding 到视觉编码

> **本章定位**：围绕视觉领域的 Encoder-only Transformer 建立 ViT 数据路径。ViT 将图像转换为 Patch Token，经双向 Encoder 建模后完成图像分类。

> **章节边界**：本章属于跨方向专题：视觉表示，以 `30` 的 Transformer Encoder 为基础，聚焦图像分类与视觉表征；多模态连接和生成式视觉模型分别由 `E20_multimodal_llm.ipynb` 与 `E50_cv_diffusion.ipynb` 承接。

**本章总览**：内容沿 Patchify、Patch Embedding、`[CLS]` 与位置编码、Encoder、分类头和预训练 ViT 微调展开，并通过标准库接口验证张量布局与模型契约。

<!-- diagram:vit-overview -->
ViT 把二维图像转换为一维 token 序列，再复用 Transformer Encoder：

![架构图：ViT 从二维图像 Patch 化到 Encoder 表征与分类 logits 的完整数据流](assets/figures/E40_cv_vit/vit-overview.svg)

[TikZ 源文件](assets/figures/E40_cv_vit/vit-overview.tex)


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 跨方向专题：视觉表示 |
| 本章定位 | 扩展到视觉 Encoder，理解图像如何转换为 Transformer Token。 |
| 先修知识 | 完成 `30`；理解卷积输入布局。 |
| 预计时间 | 90～120 分钟 |
| 运行资源 | CPU/Colab；预训练权重与图像处理器按需下载。 |
| 输入 | 批量 RGB 图像。 |
| 交付物 | 最小 ViT、标准模型配置和微调接口。 |

### 1.1．学习目标

完成本章后，读者能够解释 Patchify、Patch Embedding、`[CLS]` Token、位置编码和 Pre-LayerNorm Encoder 的数据流，并能将原理实现迁移到 torchvision 与 Transformers ViT。


### 1.2．环境与依赖

本章在 Python 3.12.13、PyTorch 2.11.0、torchvision 0.26.0 与 Transformers 5.13.1 环境下验证。预训练权重与图像处理器按需下载，原理实现路径可离线运行。


In [ ]:
# 导入本章原理实现与标准库对照所需依赖。


In [ ]:
# 固定随机状态并选择设备，创建后续视觉张量示例。

import random

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

# 3407 仅固定本章初始化、随机图像与增强序列；正式结论应使用预注册的多个 Seed，跨设备不保证逐 bit 一致。
SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")
print("PyTorch:", torch.__version__, "| device:", DEVICE)


## 2．直觉与输入输出契约

ViT 将二维图像划分为固定大小的 Patch，并把每个 Patch 投影为一维 Token。加入 `[CLS]` Token 与位置编码后，序列通过双向 Transformer Encoder，分类头读取最终 `[CLS]` 表示产生类别 Logits。

| 阶段 | 输入形状 | 输出形状 | 关键约束 |
|---|---|---|---|
| Patchify | `[B, C, H, W]` | `[B, N, C·P²]` | `H`、`W` 可被 Patch Size 整除 |
| Patch Embedding | `[B, N, C·P²]` | `[B, N, D]` | 卷积与线性投影权重可对齐 |
| 序列组装 | Patch Token | `[B, N+1, D]` | 保留 `[CLS]` 并匹配位置编码 |
| Encoder | `[B, N+1, D]` | `[B, N+1, D]` | Head 数整除隐藏维度 |
| 分类头 | `[CLS]` Hidden State | `[B, C_classes]` | 标签映射与分类头一致 |


<!-- theory-math-contract:v1 -->
### 2.1．核心机制的语言与数学表达

ViT 把二维图像变为一维 Patch Token 序列，每个 Patch 展平后线性投影到模型隐藏维度：

$$
N=\frac{H}{P_h}\frac{W}{P_w},\qquad
X_{\mathrm{patch}}\in\mathbb{R}^{B\times N\times(CP_hP_w)},\qquad
Z=X_{\mathrm{patch}}W_E+b_E\in\mathbb{R}^{B\times N\times D}
$$

其中，$B,C,H,W$ 分别是批量、通道、高和宽，$P_h,P_w$ 是 Patch 尺寸，$D$ 是隐藏维度。加入 `[CLS]` 后序列长度为 $N+1$。`my_patchify` 对应重排，`nn.Conv2d(kernel_size=stride=P)` 同时实现切分与投影。公式假设图像尺寸可被 Patch 尺寸整除；生产预处理若采用裁剪、填充或位置编码插值，必须记录不同输入分辨率的语义变化。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．Patchify：图像到 Token 序列

对 $H\times W$ 图像和 $P\times P$ patch，共有 $N=HW/P^2$ 个 token。每个 token 的原始维度是 $C P^2$。

#### 3.1.1．`reshape + permute` 的从零实现


In [ ]:
# 将图像重排为 [batch, patch_count, patch_dim] 的 Patch 序列。

def my_patchify(images: torch.Tensor, patch_size: int) -> torch.Tensor:
    """把 `[B,C,H,W]` 图像重排为扁平 Patch Token 序列。"""
    b, c, h, w = images.shape
    grid_h, grid_w = h // patch_size, w // patch_size
    # 先显式拆出网格和 Patch 内部两个空间维度。
    patches = images.reshape(
        b, c, grid_h, patch_size, grid_w, patch_size
    )
    # 将网格维移到前面，再把通道与 Patch 像素展平成一个向量。
    patches = patches.permute(0, 2, 4, 1, 3, 5).contiguous()
    return patches.view(b, grid_h * grid_w, c * patch_size**2)

# 2×3×32×32 是小型 RGB 形状夹具；分辨率增大将按面积增加 Patch Token 与 Attention 成本。
# 固定形状：images.shape = [2, 3, 32, 32]。
images = torch.arange(2 * 3 * 32 * 32, dtype=torch.float32).reshape(2, 3, 32, 32)
# 8×8 Patch 在 32×32 图像上形成 4×4=16 个 Token；Patch 越小，细节和计算量越高。
patches = my_patchify(images, patch_size=8)
print(patches.shape)


#### 3.1.2．`torch.nn.functional.unfold` 接口对照

`unfold` 是官方的滑窗提取算子；当 kernel size 等于 stride 时恰好得到互不重叠的 patch。


In [ ]:
# 使用 unfold 提取相同的非重叠 Patch，切换到标准张量算子。

lib_patches = F.unfold(images, kernel_size=8, stride=8).transpose(1, 2)
print(lib_patches.shape)


### 3.2．Patch Embedding：切分与投影

论文先展平 patch，再乘线性矩阵。工程上可用 `Conv2d(kernel_size=P, stride=P)` 等价实现，速度和内存访问更好。

#### 3.2.1．线性投影的从零实现


In [ ]:
# 把每个扁平 Patch 线性投影到 Transformer 隐藏维度。

class MyPatchEmbedding(nn.Module):
    """把固定尺寸的图像 Patch 投影到 Transformer 隐藏维度。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    def __init__(self, in_channels: int, patch_size: int, hidden_size: int):
        """创建 Patch 尺寸契约和对应的线性投影。"""
        super().__init__()
        self.patch_size = patch_size
        self.projection = nn.Linear(in_channels * patch_size**2, hidden_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """切分输入图像并返回 `[B,patch_count,hidden_size]` 表示。"""
        return self.projection(my_patchify(x, self.patch_size))

patch_embedding = MyPatchEmbedding(3, 8, 64)
tokens = patch_embedding(images)
print(tokens.shape)


#### 3.2.2．`nn.Conv2d` 等价性验证

本节复制同一组权重，验证卷积写法与“切块 + Linear”数值等价。


In [ ]:
# 用 kernel_size=stride 的卷积一次完成 Patch 切分与投影。

# 固定形状：lib_patch_embedding.weight.shape = [64, 3, 8, 8]（out_channels, in_channels/groups, kernel）。
lib_patch_embedding = nn.Conv2d(3, 64, kernel_size=8, stride=8)
# 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
with torch.no_grad():
    lib_patch_embedding.weight.copy_(patch_embedding.projection.weight.view(64, 3, 8, 8))
    lib_patch_embedding.bias.copy_(patch_embedding.projection.bias)
lib_tokens = lib_patch_embedding(images).flatten(2).transpose(1, 2)
torch.testing.assert_close(lib_tokens, tokens, rtol=1e-5, atol=1e-6)
print(lib_tokens.shape, "max error:", float((lib_tokens - tokens).abs().max().detach()))


### 3.3．`[CLS]` Token 与位置编码

自注意力不天然知道 token 顺序。ViT 为 patch token 加可学习位置向量，并在最前面拼接一个可学习的 `[CLS]` token 作为整图表示。

#### 3.3.1．序列组装的从零实现


In [ ]:
# 在 Patch 序列前加入 CLS token，并叠加可学习位置编码。

class MyTokenSequence(nn.Module):
    """在 Patch Token 前追加 CLS，并叠加可学习位置编码。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    def __init__(self, num_patches: int, hidden_size: int, dropout: float = 0.0):
        """初始化 CLS、位置参数与序列 Dropout。"""
        super().__init__()
        self.cls = nn.Parameter(torch.zeros(1, 1, hidden_size))
        self.position = nn.Parameter(torch.zeros(1, num_patches + 1, hidden_size))
        # 数据流等价性检查采用 dropout=0；训练时应按数据规模和过拟合证据重调。
        self.dropout = nn.Dropout(dropout)
        # 0.02 是 ViT/BERT 风格位置参数的小方差初始化；过大会扰动初始表示，加载预训练权重后不再生效。
        nn.init.trunc_normal_(self.position, std=0.02)

    # 前向传播按照本模块的数据流连接各子层，并返回当前阶段输出。
    def forward(self, patch_tokens: torch.Tensor) -> torch.Tensor:
        """构造包含 CLS 的完整 ViT Token 序列。"""
        cls_tokens = self.cls.expand(patch_tokens.shape[0], -1, -1)
        x = torch.cat([cls_tokens, patch_tokens], dim=1)
        return self.dropout(x + self.position[:, :x.shape[1]])

sequence_builder = MyTokenSequence(16, 64)
sequence = sequence_builder(tokens)
print(sequence.shape)


#### 3.3.2．torchvision 位置编码插值

标准 ViT 把 `class_token` 与 `encoder.pos_embedding` 注册为参数。输入分辨率变化时，加载预训练权重需对二维 patch 网格进行插值；`torchvision.models.vision_transformer.interpolate_embeddings` 已实现该逻辑。


In [ ]:
# 调用 torchvision 的位置编码插值，适配不同图像分辨率。

from torchvision.models.vision_transformer import interpolate_embeddings

# 准备持久化路径或状态对象，作为保存和重载的明确边界。
# 14×14 对应 224÷16 的预训练 Patch 网格；目标 256 会扩展到 16×16，插值时保留 CLS 位置。
lib_fake_state = {"encoder.pos_embedding": torch.randn(1, 14 * 14 + 1, 64)}
lib_resized_state = interpolate_embeddings(
    image_size=256,  # 目标分辨率改变位置网格；更换输入尺寸后须重新核对插值与显存。
    patch_size=16,
    model_state=lib_fake_state,
    interpolation_mode="bicubic",
)
print(lib_resized_state["encoder.pos_embedding"].shape)  # 1 + 16×16


### 3.4．Pre-LayerNorm Transformer Encoder

ViT 使用 Pre-LN：

$$x' = x + MSA(LN(x)), \quad x'' = x' + MLP(LN(x'))$$

相比原始 BERT 的 Post-LN，Pre-LN 在深网络中通常更易优化。

#### 3.4.1．ViT Encoder Layer 的从零实现

<!-- diagram:vit-encoder-block -->
一个 ViT Block 由两条 Pre-LayerNorm 残差子层组成：

![架构图：ViT Encoder Block 的两条 Pre-LayerNorm 残差路径](assets/figures/E40_cv_vit/vit-encoder-block.svg)

[TikZ 源文件](assets/figures/E40_cv_vit/vit-encoder-block.tex)


In [ ]:
# 用 Pre-Norm 自注意力和 MLP 组成 ViT Encoder Layer。

class MyViTEncoderLayer(nn.Module):
    """实现 ViT 的 Pre-Norm 自注意力与 MLP Encoder Layer。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    def __init__(self, hidden_size: int, num_heads: int, mlp_size: int, dropout: float = 0.0):
        """创建多头注意力、两次归一化和前馈网络。"""
        super().__init__()
        # epsilon=1e-5 沿用 PyTorch LayerNorm 默认值；加载预训练模型时必须读取 Config，低精度变化后重验稳定性。
        self.norm1 = nn.LayerNorm(hidden_size)
        self.attention = nn.MultiheadAttention(
            hidden_size, num_heads, dropout=dropout, batch_first=True
        )
        self.norm2 = nn.LayerNorm(hidden_size)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, mlp_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_size, hidden_size),
            nn.Dropout(dropout),
        )

    # 前向传播按照本模块的数据流连接各子层，并返回当前阶段输出。
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """对 Token 序列依次执行注意力和 MLP 残差更新。"""
        normed = self.norm1(x)
        attn, unused_weights = self.attention(normed, normed, normed, need_weights=False)
        x = x + attn
        return x + self.mlp(self.norm2(x))

encoder_layer = MyViTEncoderLayer(64, 4, 256)
encoded = encoder_layer(sequence)
print(encoded.shape)


#### 3.4.2．`nn.TransformerEncoderLayer` 接口对照

`norm_first=True` 对应 Pre-LN，`batch_first=True` 保持 `[B, N, D]`。


In [ ]:
# 切换到 nn.TransformerEncoderLayer，保持 batch-first 张量布局。

lib_encoder_layer = nn.TransformerEncoderLayer(
    d_model=64,
    nhead=4,
    dim_feedforward=256,
    dropout=0.0,
    activation="gelu",
    batch_first=True,
    norm_first=True,
)
lib_encoded = lib_encoder_layer(sequence)
print(lib_encoded.shape)


### 3.5．最小 ViT 集成实现

已验证的模块在此组成最小 ViT 闭环。分类头只读取最终 `[CLS]` token。该实现用于验证数据流；生产训练由经过优化并充分测试的模型库承担。


In [ ]:
# 串联 Patch Embedding、Token 序列、Encoder 和分类头得到完整 ViT。

class MyVisionTransformer(nn.Module):
    """组合 Patch 编码、Transformer Encoder 与 CLS 分类头的微型 ViT。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    # 32/8 形成 16 个 Patch；hidden=64 可被 4 头整除，4 层与 4×MLP 用于小设备机制验证。
    # 分辨率、宽度或层数增大会提高计算和显存；类别数与 RGB 三通道必须随数据契约同步。
    def __init__(
        self,
        image_size: int = 32,
        patch_size: int = 8,
        hidden_size: int = 64,
        num_heads: int = 4,
        num_layers: int = 4,
        num_classes: int = 10,
    ):
        """按图像、Patch、模型和类别配置构造完整 ViT。"""
        super().__init__()
        self.patch_embedding = MyPatchEmbedding(3, patch_size, hidden_size)
        self.sequence = MyTokenSequence((image_size // patch_size) ** 2, hidden_size)
        self.layers = nn.ModuleList([
            MyViTEncoderLayer(hidden_size, num_heads, hidden_size * 4)
            for layer_index in range(num_layers)
        ])
        self.norm = nn.LayerNorm(hidden_size)
        self.head = nn.Linear(hidden_size, num_classes)

    # 前向传播按照本模块的数据流连接各子层，并返回当前阶段输出。
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """编码图像并返回 CLS 表示对应的分类 logits。"""
        x = self.sequence(self.patch_embedding(x))
        for layer in self.layers:
            x = layer(x)
        return self.head(self.norm(x)[:, 0])

vit = MyVisionTransformer()
logits = vit(torch.randn(2, 3, 32, 32))
print(logits.shape, "parameters:", sum(p.numel() for p in vit.parameters()))


### 3.6．标准模型接口

#### 3.6.1．torchvision 结构化配置

`VisionTransformer` 适合需要明确控制网络规模、又不想维护底层实现的场景。


In [ ]:
# 用 torchvision VisionTransformer 创建结构化的从头训练模型。

from torchvision.models.vision_transformer import VisionTransformer

lib_vit_small = VisionTransformer(
    image_size=32,
    patch_size=8,
    num_layers=4,
    num_heads=4,
    hidden_dim=64,
    mlp_dim=256,
    num_classes=10,
)
# 执行前向计算，得到后续损失或解码需要的模型输出。
lib_logits = lib_vit_small(torch.randn(2, 3, 32, 32))
print(lib_logits.shape)


#### 3.6.2．Transformers 预训练权重与图像处理器

预训练模型必须配套使用同一 checkpoint 的 `AutoImageProcessor`，否则归一化、resize 或标签映射可能不一致。


In [ ]:
# 用 Transformers 图像处理器和 ViT 模型衔接真实预训练权重。

from PIL import Image
from matplotlib import cbook
from transformers import AutoImageProcessor, ViTForImageClassification

lib_checkpoint = "google/vit-base-patch16-224"
# 图像处理器从固定 Checkpoint 读取缩放与归一化；8-bit 原图通常为 0…255，浮点输入不可重复除以 255。
lib_processor = AutoImageProcessor.from_pretrained(lib_checkpoint)
lib_vit = ViTForImageClassification.from_pretrained(lib_checkpoint).to(DEVICE)
with cbook.get_sample_data("grace_hopper.jpg") as lib_image_file:
    lib_image = Image.open(lib_image_file).convert("RGB")
lib_inputs = {lib_key: lib_value.to(DEVICE) for lib_key, lib_value in lib_processor(images=lib_image, return_tensors="pt").items()}
lib_vit.eval()
# 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
with torch.inference_mode():
    lib_output = lib_vit(**lib_inputs)
lib_class_id = int(lib_output.logits.argmax(dim=-1))
print(lib_vit.config.id2label[lib_class_id], lib_output.logits.shape)


#### 3.6.3．从真实图像网格到 Patch Token 序列

本节使用 Matplotlib 随包提供的 Grace Hopper 照片，并直接消费同一 `AutoImageProcessor` 产生的 `pixel_values`，避免用与模型输入脱节的示意矩阵代替图像。

**学习问题**：图像处理器交给 ViT 的真实像素张量，如何按二维网格切分，并按行优先顺序变成 Transformer 接收的一维 Patch Token 序列？

**验收不变量**：对于处理后的 `[1, 3, 224, 224]` 图像和 `16×16` Patch，网格必须为 `14×14`，`my_patchify` 的输出必须为 `[1, 196, 768]`；网格坐标 `(row, column)` 对应的序列索引必须满足 `index = row × 14 + column`。


In [ ]:
# 直接消费预训练图像处理器产生的真实 pixel_values，呈现二维 Patch 网格到一维序列的映射。
import matplotlib.pyplot as plt

visual_patch_size = int(lib_vit.config.patch_size)
visual_pixels = lib_inputs["pixel_values"].detach().to("cpu")
visual_patches = my_patchify(visual_pixels, visual_patch_size)
_, visual_channels, visual_height, visual_width = visual_pixels.shape
visual_grid_h = visual_height // visual_patch_size
visual_grid_w = visual_width // visual_patch_size
visual_patch_count = visual_grid_h * visual_grid_w
visual_patch_dim = visual_channels * visual_patch_size**2
visual_expected_shape = (1, visual_patch_count, visual_patch_dim)
if tuple(visual_patches.shape) != visual_expected_shape:
    raise RuntimeError(
        f"Patchify 形状契约失败：期望 {visual_expected_shape}，实际 {tuple(visual_patches.shape)}"
    )
visual_probe_row, visual_probe_column = 1, 0
visual_probe_index = visual_probe_row * visual_grid_w + visual_probe_column
visual_grid_patch = visual_pixels[
    0,
    :,
    visual_probe_row * visual_patch_size:(visual_probe_row + 1) * visual_patch_size,
    visual_probe_column * visual_patch_size:(visual_probe_column + 1) * visual_patch_size,
].contiguous().reshape(-1)
visual_mapping_error = float(
    (visual_grid_patch - visual_patches[0, visual_probe_index]).abs().max()
)
if visual_mapping_error != 0.0:
    raise RuntimeError(
        f"Patch 网格到序列的行优先映射失败：索引 {visual_probe_index} 最大误差 {visual_mapping_error}"
    )

# 将标准化像素还原到可显示范围；Patch 色带只汇总每个真实 Patch 的平均 RGB。
visual_mean = torch.tensor(lib_processor.image_mean, dtype=visual_pixels.dtype)
visual_std = torch.tensor(lib_processor.image_std, dtype=visual_pixels.dtype)
visual_image = visual_pixels[0].permute(1, 2, 0) * visual_std + visual_mean
visual_image = visual_image.clamp(0, 1)
visual_patch_rgb = visual_patches.reshape(
    1, visual_patch_count, visual_channels, visual_patch_size, visual_patch_size
).mean(dim=(-1, -2))[0]
visual_patch_rgb = (visual_patch_rgb * visual_std + visual_mean).clamp(0, 1)

fig, (ax_image, ax_sequence) = plt.subplots(
    1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 1.6]}
)
ax_image.imshow(visual_image.numpy())
for boundary in range(0, visual_width + 1, visual_patch_size):
    ax_image.axvline(boundary - 0.5, color="white", linewidth=0.45, alpha=0.8)
for boundary in range(0, visual_height + 1, visual_patch_size):
    ax_image.axhline(boundary - 0.5, color="white", linewidth=0.45, alpha=0.8)
for patch_index in (0, visual_grid_w - 1, visual_grid_w, visual_patch_count - 1):
    row, column = divmod(patch_index, visual_grid_w)
    ax_image.text(
        (column + 0.5) * visual_patch_size,
        (row + 0.5) * visual_patch_size,
        str(patch_index),
        ha="center",
        va="center",
        color="black",
        fontsize=9,
        bbox={"facecolor": "white", "alpha": 0.82, "edgecolor": "none"},
    )
ax_image.set_title(f"模型实际输入：{visual_grid_h}×{visual_grid_w} Patch 网格")
ax_image.set_axis_off()

ax_sequence.imshow(visual_patch_rgb.unsqueeze(0).numpy(), aspect="auto")
sequence_ticks = [0, 1, visual_grid_w - 1, visual_grid_w, visual_grid_w + 1, visual_patch_count - 1]
ax_sequence.set_xticks(sequence_ticks)
ax_sequence.set_xticklabels([str(index) for index in sequence_ticks])
ax_sequence.set_yticks([0])
ax_sequence.set_yticklabels(["Patch 平均 RGB"])
ax_sequence.set_xlabel("行优先 Patch Token 索引")
ax_sequence.set_title(
    f"二维网格展平后的序列：{tuple(visual_patches.shape)}；每个色块代表一个 {visual_patch_dim} 维 Patch 向量"
)
ax_sequence.text(
    0.5,
    -0.42,
    f"(row, column) → index = row × {visual_grid_w} + column",
    transform=ax_sequence.transAxes,
    ha="center",
    va="top",
)
fig.suptitle("真实图像的 Patchify：空间网格转换为 Token 序列", fontsize=14)
fig.tight_layout()
plt.show()

print({
    "pixel_values": tuple(visual_pixels.shape),
    "patch_grid": (visual_grid_h, visual_grid_w),
    "patch_sequence": tuple(visual_patches.shape),
    "last_patch_index": (visual_grid_h - 1) * visual_grid_w + (visual_grid_w - 1),
    "row_major_probe": (visual_probe_row, visual_probe_column, visual_probe_index),
    "row_major_max_error": visual_mapping_error,
})


**应观察结论**：左图中的 Patch `0` 到 `13` 构成第一行，Patch `14` 从第二行重新开始；右图按照同一行优先顺序排列 `196` 个 Patch。每个序列元素仍包含 `3×16×16=768` 个像素分量，后续 Patch Embedding 才把它投影到隐藏维度。

**不可误读边界**：右侧色带仅用每个 Patch 的平均 RGB 帮助识别顺序，不是 Patch Embedding、注意力权重或语义重要性；网格映射正确也不能单独证明分类结果正确。


#### 3.6.4．Patch Size 是视觉 Token 的光圈

**学习问题。** 对同一张模型实际输入图像，Patch Size 从 `32` 缩小到 `16`、再缩小到 `8` 时，空间网格、Token 序列和标准全注意力的逻辑矩阵规模如何联动变化？

本分镜始终复用 `lib_inputs["pixel_values"]`，只改变 `my_patchify` 的 Patch Size。对于形状 `[B,C,H,W]` 的图像，Patch Token 数为 $N=(H/P)(W/P)$；加入一个 `[CLS]` 后，Encoder 序列长度为 $L=N+1$，每层每个 Head 的标准全注意力逻辑矩阵包含 $L^2$ 个元素。Patch Size 每减半，Patch Token 数在当前二维图像上扩大为 4 倍，而不计 `[CLS]` 时的逻辑矩阵元素数扩大为 16 倍。

运行前检查以下不变量：

- `224×224` 输入必须能被 `P=32/16/8` 整除，三组网格分别为 `7×7`、`14×14`、`28×28`。
- 每个 Patch Size 下，`my_patchify` 必须与 `F.unfold(kernel_size=P, stride=P)` 在相同顺序上数值一致，并通过首行末尾、第二行开头和最后 Patch 的行优先映射探针。
- 真实像素与 Patch 张量在绘图前统一沿 `detach() → float() → cpu()` 转换；序列色带只显示有限数量的开头 Token，完整数量与形状同时输出。


In [ ]:
# 对同一真实图像改变 Patch Size，核对网格、序列顺序和标准全注意力逻辑规模。
import matplotlib.pyplot as plt

PATCH_APERTURE_SIZES = (32, 16, 8)
PATCH_TOKEN_DISPLAY_LIMIT = 20  # 只显示序列开头，避免把 784 个 Patch 压缩成不可读色带。
PATCH_UNFOLD_RTOL = 1e-5  # FP32 原理路径与标准算子对照的相对容差。
PATCH_UNFOLD_ATOL = 1e-6  # 接近零的标准化像素使用绝对容差。
aperture_pixels = lib_inputs["pixel_values"].detach().float().cpu()
_, aperture_channels, aperture_height, aperture_width = aperture_pixels.shape
aperture_mean = torch.tensor(lib_processor.image_mean, dtype=aperture_pixels.dtype)
aperture_std = torch.tensor(lib_processor.image_std, dtype=aperture_pixels.dtype)
aperture_image = (
    aperture_pixels[0].permute(1, 2, 0) * aperture_std + aperture_mean
).clamp(0, 1)
aperture_records = []

for patch_size in PATCH_APERTURE_SIZES:
    if aperture_height % patch_size != 0 or aperture_width % patch_size != 0:
        raise RuntimeError(
            f"输入 {aperture_height}×{aperture_width} 不能按 P={patch_size} 无重叠切分"
        )
    grid_h = aperture_height // patch_size
    grid_w = aperture_width // patch_size
    patch_count = grid_h * grid_w
    principle_patches = my_patchify(aperture_pixels, patch_size).detach().float().cpu()
    unfold_patches = F.unfold(
        aperture_pixels, kernel_size=patch_size, stride=patch_size
    ).transpose(1, 2).detach().float().cpu()
    expected_shape = (
        1, patch_count, aperture_channels * patch_size**2
    )
    if tuple(principle_patches.shape) != expected_shape:
        raise RuntimeError(
            f"P={patch_size} 的 Patchify 形状失败：期望 {expected_shape}，实际 {tuple(principle_patches.shape)}"
        )
    torch.testing.assert_close(
        principle_patches, unfold_patches,
        rtol=PATCH_UNFOLD_RTOL, atol=PATCH_UNFOLD_ATOL,
    )
    unfold_max_error = float((principle_patches - unfold_patches).abs().max())

    probe_indices = sorted(set((0, grid_w - 1, grid_w, patch_count - 1)))
    row_major_errors = []
    for patch_index in probe_indices:
        row, column = divmod(patch_index, grid_w)
        grid_patch = aperture_pixels[
            0, :,
            row * patch_size:(row + 1) * patch_size,
            column * patch_size:(column + 1) * patch_size,
        ].contiguous().reshape(-1)
        row_major_errors.append(float(
            (grid_patch - principle_patches[0, patch_index]).abs().max()
        ))
    row_major_max_error = max(row_major_errors)
    if row_major_max_error != 0.0:
        raise RuntimeError(
            f"P={patch_size} 的网格到序列行优先映射失败：最大误差 {row_major_max_error}"
        )

    patch_rgb = principle_patches.reshape(
        1, patch_count, aperture_channels, patch_size, patch_size
    ).mean(dim=(-1, -2))[0]
    patch_rgb = (patch_rgb * aperture_std + aperture_mean).clamp(0, 1)
    sequence_length = patch_count + 1  # 标准 ViT 在 Patch Token 前加入一个 CLS Token。
    attention_elements = sequence_length**2
    aperture_records.append({
        "patch_size": patch_size,
        "grid_h": grid_h,
        "grid_w": grid_w,
        "patch_count": patch_count,
        "patch_dim": aperture_channels * patch_size**2,
        "patches": principle_patches,
        "patch_rgb": patch_rgb,
        "sequence_length": sequence_length,
        "attention_elements": attention_elements,
        "displayed_tokens": min(PATCH_TOKEN_DISPLAY_LIMIT, patch_count),
        "unfold_max_error": unfold_max_error,
        "row_major_probe_indices": probe_indices,
        "row_major_max_error": row_major_max_error,
    })

patch_counts = [record["patch_count"] for record in aperture_records]
attention_counts = [record["attention_elements"] for record in aperture_records]
if not all(left < right for left, right in zip(patch_counts, patch_counts[1:])):
    raise RuntimeError("Patch Size 减小时 Patch Token 数未按预期增长")
if not all(left < right for left, right in zip(attention_counts, attention_counts[1:])):
    raise RuntimeError("Patch Size 减小时全注意力逻辑矩阵未按预期增长")

figure = plt.figure(figsize=(15, 9), constrained_layout=True)
layout = figure.add_gridspec(3, 3, height_ratios=(3.0, 0.65, 1.55))

for column, record in enumerate(aperture_records):
    patch_size = record["patch_size"]
    grid_h, grid_w = record["grid_h"], record["grid_w"]
    patch_count = record["patch_count"]
    image_axis = figure.add_subplot(layout[0, column])
    image_axis.imshow(aperture_image.numpy())
    for boundary in range(0, aperture_width + 1, patch_size):
        image_axis.axvline(boundary - 0.5, color="white", linewidth=0.35, alpha=0.75)
    for boundary in range(0, aperture_height + 1, patch_size):
        image_axis.axhline(boundary - 0.5, color="white", linewidth=0.35, alpha=0.75)
    for patch_index in record["row_major_probe_indices"]:
        row, patch_column = divmod(patch_index, grid_w)
        image_axis.text(
            (patch_column + 0.5) * patch_size,
            (row + 0.5) * patch_size,
            str(patch_index), ha="center", va="center", fontsize=8, color="black",
            bbox={"facecolor": "white", "alpha": 0.82, "edgecolor": "none"},
        )
    image_axis.set_title(
        f"P={patch_size}：{grid_h}×{grid_w} 网格 → N={patch_count}"
    )
    image_axis.set_axis_off()

    sequence_axis = figure.add_subplot(layout[1, column])
    display_count = record["displayed_tokens"]
    sequence_axis.imshow(
        record["patch_rgb"][:display_count].unsqueeze(0).numpy(), aspect="auto"
    )
    sequence_ticks = sorted(set((
        0, min(grid_w - 1, display_count - 1), display_count - 1
    )))
    sequence_axis.set_xticks(sequence_ticks)
    sequence_axis.set_xticklabels([str(index) for index in sequence_ticks])
    sequence_axis.set_yticks([])
    sequence_axis.set_title(
        f"仅显示前 {display_count}/{patch_count} 个行优先 Patch Token", fontsize=10
    )

summary_axis = figure.add_subplot(layout[2, :])
x_positions = np.arange(len(aperture_records))
bar_width = 0.34
sequence_lengths = [record["sequence_length"] for record in aperture_records]
attention_elements = [record["attention_elements"] for record in aperture_records]
sequence_bars = summary_axis.bar(
    x_positions - bar_width / 2, sequence_lengths, width=bar_width,
    color="#0072B2", label="序列 Token：L=N+1",
)
attention_bars = summary_axis.bar(
    x_positions + bar_width / 2, attention_elements, width=bar_width,
    color="#E69F00", label="标准全注意力元素：L²",
)
summary_axis.set_yscale("log")
summary_axis.set(
    title="同一图像：序列线性增长，标准全注意力逻辑矩阵按序列长度平方增长",
    xlabel="Patch Size", ylabel="数量（对数刻度）",
    xticks=x_positions,
    xticklabels=[f"P={record['patch_size']}\nN={record['patch_count']}" for record in aperture_records],
)
summary_axis.bar_label(sequence_bars, labels=[f"{value:,}" for value in sequence_lengths], padding=3)
summary_axis.bar_label(attention_bars, labels=[f"{value:,}" for value in attention_elements], padding=3)
summary_axis.grid(axis="y", alpha=0.25, which="both")
summary_axis.legend()
figure.suptitle("Patch Size 是视觉 Token 的光圈：同一图像，不同序列与注意力规模", fontsize=14)
plt.show()

aperture_summary = [
    {
        "patch_size": record["patch_size"],
        "patch_grid": (record["grid_h"], record["grid_w"]),
        "patch_sequence_shape": tuple(record["patches"].shape),
        "patch_tokens": record["patch_count"],
        "sequence_tokens_with_cls": record["sequence_length"],
        "attention_matrix_shape_per_head": (record["sequence_length"], record["sequence_length"]),
        "attention_elements_per_head": record["attention_elements"],
        "displayed_patch_tokens": record["displayed_tokens"],
        "unfold_max_error": record["unfold_max_error"],
        "row_major_max_error": record["row_major_max_error"],
    }
    for record in aperture_records
]
print({
    "pixel_values": tuple(aperture_pixels.shape),
    "configurations": aperture_summary,
    "patch_token_growth": [
        patch_counts[index + 1] / patch_counts[index]
        for index in range(len(patch_counts) - 1)
    ],
    "attention_element_growth_with_cls": [
        attention_counts[index + 1] / attention_counts[index]
        for index in range(len(attention_counts) - 1)
    ],
})


**应观察结论**：三幅分镜中的人物、裁剪范围和标准化输入保持不变，只有白色网格随 Patch Size 缩小而变密。`P=32/16/8` 分别产生 `49/196/784` 个 Patch Token；加入 `[CLS]` 后序列长度为 `50/197/785`，每层每个 Head 的标准全注意力逻辑矩阵分别包含 `2,500/38,809/616,225` 个元素。忽略单个 `[CLS]` 的微小影响，每次将 Patch Size 减半都会使 Token 数扩大 4 倍、逻辑注意力元素数扩大 16 倍。`F.unfold` 和行优先探针同时证明三条色带来自同一真实像素张量，而不是手填的规模示意。

**不可误读边界**：序列色带只显示每个 Patch 的平均 RGB 和有限开头位置，不是 Patch Embedding、Attention 权重或语义贡献。`P=32` 与 `P=8` 在这里是张量布局和容量反事实；当前 `vit-base-patch16-224` Checkpoint 的 Patch 投影权重与位置编码按 `P=16` 训练，不能直接把另外两种切分送入该预训练模型并期待有效分类。$L^2$ 描述标准全注意力的逻辑分数数量；FlashAttention 等 Kernel 可能不显式物化完整矩阵，实际显存、吞吐和延迟仍需在目标硬件测量。更小 Patch 保留更多局部细节，但不保证模型质量单调提高。


### 3.7．下游微调与分类头适配

真实数据集应划分 train/validation/test，训练阶段做随机裁剪和翻转，验证阶段使用确定性 resize/crop。本节给出模型与优化器的标准连接方式。

<!-- diagram:vit-finetune-lifecycle -->
迁移到真实任务时，应保持预训练主干、图像预处理和新分类头的契约一致：

```mermaid
flowchart LR
    P["预训练 ViT 权重"] --> B["加载 Encoder 主干"]
    D["目标图像数据"] --> R["Resize / Normalize"]
    R --> B
    B --> H["替换分类头"]
    H --> F["分阶段解冻并微调"]
    F --> E["验证集选择 checkpoint"]
    E --> A["保存权重 + 配置 + 预处理参数"]
```


In [ ]:
# 替换分类头并只暴露新任务需要训练的参数。

from torchvision.models import ViT_B_16_Weights, vit_b_16

lib_weights = ViT_B_16_Weights.IMAGENET1K_V1
lib_model = vit_b_16(weights=lib_weights)
lib_model.heads.head = nn.Linear(lib_model.heads.head.in_features, 10)
lib_model = lib_model.to(DEVICE)
# 3e-5/0.05 是预训练 ViT 微调的保守起点；须随有效 Batch、训练轮数和验证指标联合复核。
# betas 与 epsilon 显式落盘以固定优化器契约，不能把库默认值视为算法常量。
lib_optimizer = torch.optim.AdamW(
    lib_model.parameters(),
    lr=3e-5, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.05,
)
lib_transforms = lib_weights.transforms()
print(lib_model.heads, lib_transforms)


## 4．证据验证

1. Patch Size 从 16 调整为 8 后，Token 数应扩大为原来的 4 倍，注意力矩阵元素数扩大为原来的 16 倍。
2. `reshape + permute`、`unfold` 与 `Conv2d` 路径在共享权重时应产生一致的 Patch 表示。
3. 分辨率变化时，位置编码插值应保留 `[CLS]` 位置，并在二维 Patch 网格上进行。
4. 原理实现与 `nn.TransformerEncoderLayer` 的输入输出形状、Pre-LN 语义和分类 Logits 维度应一致。
5. 记录图像处理器 ID、模型 ID、文件哈希与标签映射后，保存加载结果应保持一致。


## 5．迁移到生产库

| 原理对象 | 生产库对象 | 迁移重点 |
|---|---|---|
| Patch 切分 | `torch.nn.functional.unfold` | 张量布局与 Patch 顺序 |
| Patch Embedding | `nn.Conv2d` | Kernel、Stride 与权重映射 |
| Pre-LN Encoder | `nn.TransformerEncoderLayer` | `norm_first`、`batch_first` 与 Dropout |
| 完整 ViT | torchvision `VisionTransformer` | 结构配置与位置编码插值 |
| 预训练模型 | Transformers `ViTForImageClassification` | `AutoImageProcessor`、标签映射与文件哈希 |

生产路径应绑定模型权重、图像处理器、Config 与标签映射；原理实现保留为张量语义和等价性回归基线。


## 6．生产边界

1. 通道顺序、像素范围、Resize、Crop 与归一化参数应与预训练检查点一致。
2. 输入分辨率需要满足 Patch 划分约束；位置编码插值应保留二维空间结构与 `[CLS]` 位置。
3. 小数据集从头训练大型 ViT 容易过拟合，生产微调应明确冻结策略、增强配方、学习率与验证预处理。
4. 混合精度、梯度累积、EMA、分层学习率衰减和数据增强应通过消融与固定验证集评估。
5. 发布产物应包含模型/处理器 ID、文件哈希、标签映射、质量指标、性能数据和恢复路径。
